In [2]:
import pandas as pd

# 读取csv文件
chexpert_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-2.0.0-chexpert.csv')
metadata_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-2.0.0-metadata.csv')

# 打印head查看数据结构
chexpert_head = chexpert_df.head()
metadata_head = metadata_df.head()

chexpert_head, metadata_head

(   subject_id  study_id  Atelectasis  Cardiomegaly  Consolidation  Edema  \
 0    10000032  50414267          NaN           NaN            NaN    NaN   
 1    10000032  53189527          NaN           NaN            NaN    NaN   
 2    10000032  53911762          NaN           NaN            NaN    NaN   
 3    10000032  56699142          NaN           NaN            NaN    NaN   
 4    10000764  57375967          NaN           NaN            1.0    NaN   
 
    Enlarged Cardiomediastinum  Fracture  Lung Lesion  Lung Opacity  \
 0                         NaN       NaN          NaN           NaN   
 1                         NaN       NaN          NaN           NaN   
 2                         NaN       NaN          NaN           NaN   
 3                         NaN       NaN          NaN           NaN   
 4                         NaN       NaN          NaN           NaN   
 
    No Finding  Pleural Effusion  Pleural Other  Pneumonia  Pneumothorax  \
 0         1.0               NaN

In [3]:
# 读取第三个表格（mscxr数据集）
mscxr_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/MS_CXR_Local_Alignment_v1.0.0.csv')

# 打印前五行数据来查看结构
mscxr_head = mscxr_df.head()

# 打印列名
mscxr_columns = mscxr_df.columns.tolist()

mscxr_head, mscxr_columns


(                                       dicom_id category_name  \
 0  675d792f-a3521e48-5eec8573-1e81d644-e60c34f8     Pneumonia   
 1  675d792f-a3521e48-5eec8573-1e81d644-e60c34f8     Pneumonia   
 2  5318d353-daae9c3d-2ee8648e-32b65198-aeff801e     Pneumonia   
 3  5318d353-daae9c3d-2ee8648e-32b65198-aeff801e     Pneumonia   
 4  4decce85-c6ede74e-7a8bc81c-e81edee9-5ec17116  Pneumothorax   
 
                                     label_text  \
 0                          Bibasilar opacities   
 1                          Bibasilar opacities   
 2  Bilateral multifocal areas of consolidation   
 3  Bilateral multifocal areas of consolidation   
 4               Large right-sided pneumothorax   
 
                                                 path     x     y    w     h  \
 0  files/p10/p10233088/s54276838/675d792f-a3521e4...   196  1136  532   315   
 1  files/p10/p10233088/s54276838/675d792f-a3521e4...  1009  1134  491   350   
 2  files/p10/p10123147/s50230934/5318d353-daae9c3... 

In [7]:
import pandas as pd

# 读取第三个表格（mscxr数据集）
mscxr_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/MS_CXR_Local_Alignment_v1.0.0.csv')

# 读取前两个表格
chexpert_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-2.0.0-chexpert.csv')
metadata_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-2.0.0-metadata.csv')

# 1. 从 metadata_df 获取 subject_id, study_id, ViewPosition 对应 dicom_id
metadata_info = metadata_df[['dicom_id', 'subject_id', 'study_id', 'ViewPosition']]

# 2. 合并 ms_cxr 数据集和 metadata 信息
mscxr_with_metadata = pd.merge(mscxr_df, metadata_info, how='left', on='dicom_id')

# 3. 从 chexpert_df 获取疾病信息，使用 subject_id 和 study_id 来查找
# 先确保 'subject_id' 和 'study_id' 在 chexpert_df 中存在
chexpert_relevant = chexpert_df[['subject_id', 'study_id'] + chexpert_df.columns.tolist()[2:]]

# 4. 合并数据：将 mscxr_with_metadata 和 chexpert_relevant 进行合并
final_df = pd.merge(mscxr_with_metadata, chexpert_relevant, how='left', on=['subject_id', 'study_id'])

# 打印最终合并表格的前五行
final_df.iloc[1, 3], final_df.columns.tolist()


('files/p10/p10233088/s54276838/675d792f-a3521e48-5eec8573-1e81d644-e60c34f8.jpg',
 ['dicom_id',
  'category_name',
  'label_text',
  'path',
  'x',
  'y',
  'w',
  'h',
  'image_width',
  'image_height',
  'subject_id',
  'study_id',
  'ViewPosition',
  'Atelectasis',
  'Cardiomegaly',
  'Consolidation',
  'Edema',
  'Enlarged Cardiomediastinum',
  'Fracture',
  'Lung Lesion',
  'Lung Opacity',
  'No Finding',
  'Pleural Effusion',
  'Pleural Other',
  'Pneumonia',
  'Pneumothorax',
  'Support Devices'])

In [6]:
# 保存最终合并的表格
final_df.to_csv('/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/MS_CXR_combined.csv', index=False)

# 统计 ViewPosition 每个值的数量
view_position_counts = final_df['ViewPosition'].value_counts()

view_position_counts


ViewPosition
AP    1125
PA     323
Name: count, dtype: int64

In [12]:
import os
import shutil
import pandas as pd
from multiprocessing import Pool, cpu_count

# 定义拷贝文件的函数
def copy_file(row):
    prefix = "/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/"
    target_base_dir = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/'

    source_path = prefix + row['path']
    dicom_id = row['dicom_id']

    target_dir = os.path.join(target_base_dir, dicom_id)
    
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    
    target_path = os.path.join(target_dir, 'original.jpg')

    if os.path.exists(source_path):
        shutil.copy(source_path, target_path)
    else:
        print(f"文件不存在: {source_path}")

# 读取最终合并的表格
final_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/MS_CXR_combined.csv')

# 使用 CPU 核心数来决定进程数
num_processes = cpu_count()

# 创建进程池并执行文件拷贝任务
with Pool(processes=num_processes) as pool:
    pool.map(copy_file, [row for index, row in final_df.iterrows()])

print("文件拷贝完成！")


文件拷贝完成！


In [15]:
import pandas as pd

# 读取数据
metadata_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-2.0.0-metadata.csv')
mscxr_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/MS_CXR_Local_Alignment_v1.0.0.csv')
chexpert_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-2.0.0-chexpert.csv')

# 1. 获取metadata_df中的dicom_id列
metadata_dicom_ids = set(metadata_df['dicom_id'])

# 2. 获取mscxr_df中的dicom_id列
mscxr_dicom_ids = set(mscxr_df['dicom_id'])

# 3. 得到剩余的dicom_id（即在metadata中但不在mscxr中）
remaining_dicom_ids = metadata_dicom_ids - mscxr_dicom_ids

# 4. 从metadata_df中筛选出这些剩余的dicom_id，并且ViewPosition为"AP"或"PA"
filtered_metadata = metadata_df[metadata_df['dicom_id'].isin(remaining_dicom_ids)]
filtered_metadata = filtered_metadata[filtered_metadata['ViewPosition'].isin(['AP', 'PA'])]

# 5. 从chexpert_df中提取与filtered_metadata中相同subject_id和study_id的记录
merged_data = pd.merge(filtered_metadata, chexpert_df, how='left', on=['subject_id', 'study_id'])

# 6. 保存结果为CXV文件（如果你需要保存为CSV格式，可以修改扩展名为.csv）
merged_data.to_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-remained-data-ap-pa-without-ms-cxr-part.csv', index=False)

print("表格已保存！")


表格已保存！


In [17]:
import pandas as pd

# 读取数据
remained_data = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-remained-data-ap-pa-without-ms-cxr-part.csv')
mscxr_data = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/MS_CXR_Local_Alignment_v1.0.0.csv')

# 提取 dicom_id 列
remained_dicom_ids = set(remained_data['dicom_id'])
mscxr_dicom_ids = set(mscxr_data['dicom_id'])

# 检查两个 dicom_id 列是否有交集
intersection = remained_dicom_ids.intersection(mscxr_dicom_ids)

# 打印交集的大小，如果没有交集，则返回空集合
if len(intersection) == 0:
    print("没有交集，测试通过！")
else:
    print(f"有交集，交集大小: {len(intersection)}")
    print("交集的 dicom_id:", intersection)


没有交集，测试通过！


In [19]:
# 读取数据
remained_data = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-remained-data-ap-pa-without-ms-cxr-part.csv')

# 统计 dicom_id 的数量
total_dicom_count = remained_data['dicom_id'].nunique()

total_dicom_count


242287

In [21]:
import pandas as pd
import numpy as np

# 读取数据
remained_data = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-remained-data-ap-pa-without-ms-cxr-part.csv')

# 设置随机种子，以确保结果可复现
np.random.seed(42)

# 计算总数据量
total_data_count = remained_data.shape[0]
print(f"总共有 {total_data_count} 张数据")

# 1. 抽取5个不重合的测试集（5-fold交叉验证）
test_data_folds = []
remaining_data = remained_data.copy()

# 每次从剩余数据中随机抽取20%作为测试集
for fold in range(5):
    test_size = total_data_count // 5
    test_data = remaining_data.sample(n=test_size, random_state=42 + fold)  # 不同fold使用不同的随机种子
    test_data_folds.append(test_data)
    remaining_data = remaining_data.drop(test_data.index)  # 从剩余数据中删除已抽取的测试集

# 保存每个fold的测试集
for fold in range(5):
    test_data_folds[fold].to_csv(f'/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold{fold}_test.csv', index=False)
    print(f"已保存 fold{fold}_test.csv, 包含 {test_data_folds[fold].shape[0]} 张数据")

# 2. 对于每个 fold，从剩余的训练集中抽取不同大小的子集
for fold in range(5):
    # 剩余的训练数据（去除当前 fold 的测试集）
    train_data_remaining = remained_data.drop(test_data_folds[fold].index)

    # 定义不同大小的子集
    subset_sizes = [1000, 5000, 10000, 20000, 50000, 100000]
    start_index = 0

    # 生成子集并保存
    for size in subset_sizes:
        # 从剩余数据中随机抽取指定数量的数据
        subset = train_data_remaining.sample(n=size, random_state=42 + fold + size)  # 每次使用不同的随机种子
        # 将之前的子集合并
        if start_index == 0:
            final_subset = subset
        else:
            final_subset = pd.concat([final_subset, subset], axis=0)
        
        # 更新剩余数据
        train_data_remaining = train_data_remaining.drop(subset.index)
        
        # 保存当前子集
        final_subset.to_csv(f'/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold{fold}_train_subset_{size}.csv', index=False)
        
        print(f"已保存 fold{fold}_train_subset_{size}.csv, 包含 {final_subset.shape[0]} 张数据")

print("所有折的子集已保存完成！")


总共有 242287 张数据
已保存 fold0_test.csv, 包含 48457 张数据
已保存 fold1_test.csv, 包含 48457 张数据
已保存 fold2_test.csv, 包含 48457 张数据
已保存 fold3_test.csv, 包含 48457 张数据
已保存 fold4_test.csv, 包含 48457 张数据
已保存 fold0_train_subset_1000.csv, 包含 1000 张数据
已保存 fold0_train_subset_5000.csv, 包含 5000 张数据
已保存 fold0_train_subset_10000.csv, 包含 10000 张数据
已保存 fold0_train_subset_20000.csv, 包含 20000 张数据
已保存 fold0_train_subset_50000.csv, 包含 50000 张数据
已保存 fold0_train_subset_100000.csv, 包含 100000 张数据
已保存 fold1_train_subset_1000.csv, 包含 1000 张数据
已保存 fold1_train_subset_5000.csv, 包含 5000 张数据
已保存 fold1_train_subset_10000.csv, 包含 10000 张数据
已保存 fold1_train_subset_20000.csv, 包含 20000 张数据
已保存 fold1_train_subset_50000.csv, 包含 50000 张数据
已保存 fold1_train_subset_100000.csv, 包含 100000 张数据
已保存 fold2_train_subset_1000.csv, 包含 1000 张数据
已保存 fold2_train_subset_5000.csv, 包含 5000 张数据
已保存 fold2_train_subset_10000.csv, 包含 10000 张数据
已保存 fold2_train_subset_20000.csv, 包含 20000 张数据
已保存 fold2_train_subset_50000.csv, 包含 50000 张数据
已保存 fold2_train_subset_100000

In [20]:
import os
import pandas as pd
from multiprocessing import Pool, cpu_count

# 假设你的目录结构已经整理好
prefix = "/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org"
base_directory = "/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold_data"
folds = [0, 1, 2, 3, 4]  # 根据需要处理的fold范围

divs = ["train_subset_1000", "train_subset_5000", "train_subset_10000", "train_subset_20000", "train_subset_50000", "train_subset_100000"]

# 函数来构造路径
def construct_file_path(dicom_id, subject_id, study_id):
    try:
        # 提取 subject_id 前两位，形成 p{subject_id}
        subject_prefix = f"p{str(subject_id)[:2]}"  # 假设 subject_id 的前两位作为目录
        # 构建完整路径
        path = os.path.join(prefix, f"files/{subject_prefix}/p{subject_id}/s{study_id}/{dicom_id}.jpg")
        # print(f"path:{path}")
        if os.path.exists(path):
            return dicom_id, path  # 返回dicom_id和找到的路径
        else:
            raise FileNotFoundError(f"文件不存在: {path}")
    except Exception as e:
        return dicom_id, str(e)  # 返回dicom_id和错误信息

# 多进程处理函数
def process_fold(fold, div):
    print(f"\n正在处理 fold{fold}...")

    # 读取fold的原始数据文件
    fold_file = f'/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold{fold}/fold{fold}_{div}.csv'
    fold_data = pd.read_csv(fold_file)

    # 使用多进程查找每个 dicom_id 的路径
    with Pool(processes=cpu_count()) as pool:
        results = pool.starmap(construct_file_path, 
                               [(row['dicom_id'], row['subject_id'], row['study_id']) for index, row in fold_data.iterrows()])
    
    # 将结果添加到DataFrame中
    paths = {dicom_id: path for dicom_id, path in results}
    fold_data['path'] = fold_data['dicom_id'].map(paths)
    
    # 过滤出找不到路径的记录，输出错误并跳过
    missing_paths = fold_data[fold_data['path'].str.contains('文件不存在')]
    if not missing_paths.empty:
        print(f"警告: 以下记录找不到文件路径，并将被跳过：")
        print(missing_paths[['dicom_id', 'subject_id', 'study_id', 'path']])
        # 从数据中删除找不到路径的行
        fold_data = fold_data[~fold_data['path'].str.contains('文件不存在')]

    # 创建新目录并保存处理后的文件
    new_fold_dir = os.path.join(base_directory, f"fold{fold}")
    os.makedirs(new_fold_dir, exist_ok=True)

    new_file_path = os.path.join(new_fold_dir, f"fold{fold}_{div}.csv")
    fold_data.to_csv(new_file_path, index=False)
    print(f"处理完毕，新的数据已保存至 {new_file_path}")

# 遍历每个fold进行处理
for fold in folds:
    for div in divs:
        process_fold(fold, div)

print("所有fold处理完成！")



正在处理 fold0...


FileNotFoundError: [Errno 2] No such file or directory: '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold0/fold0_train_subset_1000.csv'

In [29]:
import os
from PIL import Image

# 定义根目录
base_dir = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/'

# 遍历一级子文件夹
for subdir in os.listdir(base_dir):
    subdir_path = os.path.join(base_dir, subdir)
    
    # 确保是文件夹
    if os.path.isdir(subdir_path):
        original_path = os.path.join(subdir_path, 'original.jpg')
        
        # 检查 original.jpg 是否存在
        if os.path.exists(original_path):
            try:
                # 打开原始图像
                with Image.open(original_path) as img:
                    # 调整图像大小到 256x256
                    img_resized = img.resize((256, 256))
                    
                    # 保存调整后的图像
                    resized_path = os.path.join(subdir_path, 'original_256.jpg')
                    img_resized.save(resized_path)
                    print(f"图像已调整并保存: {resized_path}")
            except Exception as e:
                print(f"处理 {original_path} 时发生错误: {e}")
        else:
            print(f"未找到文件: {original_path}")


图像已调整并保存: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/f7ba6691-53545537-20c8b2dc-79dbd392-36f05d15/original_256.jpg
图像已调整并保存: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/1d3cf33d-0bcbe0fd-589cde2e-ff4cd9b4-41b8ed96/original_256.jpg
图像已调整并保存: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/0ef420a4-7f73b19a-2b43ee81-f8f3e1ed-d94b9a27/original_256.jpg
图像已调整并保存: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/f7b69ee3-db7f264c-fca7d1c7-1d372fc0-02b35a47/original_256.jpg
图像已调整并保存: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/36a8ff8b-be622816-dcc0c794-e85c285d-9fde04b4/original_256.jpg
图像已调整并保存: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/a5d3aa8b-573aa054-cb848da6-184d98fa-539aecf3/original_256.jpg
图像已调整并保存: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/1e16b6b4-d5a7ebf8-78fae5b1-fa122b81-cb68fb7c/original_256.jpg
图像已调整并保存: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_images/09a6c621-8faee2f2-bd72a98f-ed8bee84-9767b0a9/o

In [33]:
import pandas as pd
import numpy as np
import os

# 读取 metadata 和 chexpert 数据
metadata_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-2.0.0-metadata.csv')
chexpert_df = pd.read_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-2.0.0-chexpert.csv')

# 1. 合并 metadata 和 chexpert 数据
merged_df = pd.merge(metadata_df, chexpert_df, how='left', on=['subject_id', 'study_id'])

# 打印合并前后的行数
print(f"合并前的行数: {metadata_df.shape[0]}")
print(f"合并后的行数: {merged_df.shape[0]}")

# 2. 只保留指定的疾病记录
disease_columns = ['Cardiomegaly', 'Consolidation', 'Edema', 'Lung Lesion', 'Lung Opacity', 'Pleural Effusion', 'Pneumonia', 'Pneumothorax']

# 3. 将疾病列中的 1.0 转换为 YES，0.0 转换为 NO，其他值和 nan 忽略
for disease in disease_columns:
    merged_df[disease] = merged_df[disease].apply(lambda x: 'YES' if x == 1.0 else ('NO' if x == 0.0 else np.nan))

# 4. 拆分数据，每个疾病一行，删除 NaN 和 -1
expanded_rows = []

# 定义路径前缀
prefix = "/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files"
prefix_256 = "/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files_256"

def construct_image_paths(dicom_id, subject_id, study_id):
    # 构造原始图像路径
    subject_prefix = f"p{str(subject_id)[:2]}"  # 假设 subject_id 的前两位作为目录
    img_original_path = os.path.join(prefix, f"{subject_prefix}/p{subject_id}/s{study_id}/{dicom_id}.jpg")
    
    # 构造 256 图像路径
    img_256_path = os.path.join(prefix_256, f"{subject_prefix}/p{subject_id}/s{study_id}/{dicom_id}.jpg")
    
    return img_original_path, img_256_path

# 生成唯一的 seed，从 0 开始递增
seed_counter = 0

for index, row in merged_df.iterrows():
    subject_id = row['subject_id']
    study_id = row['study_id']
    dicom_id = row['dicom_id']
    
    # 遍历所有疾病列，判断该疾病是否为 YES
    for disease in disease_columns:
        if row[disease] in ['YES', 'NO']:
            exists = row[disease]
            img_original_path, img_256_path = construct_image_paths(dicom_id, subject_id, study_id)
            
            expanded_rows.append({
                'seed': seed_counter,  # 使用递增的 seed
                'data_source': "cxr_crop",
                'prompt': [{"content": "", "role": "user"}],
                'extra_info': {
                    "env_config": {"parquet_path": "data/cxr_all/full.parquet"},
                    "env_name": "cxr_crop",
                    "seed": index,
                    "split": "train"
                },
                'dicom_id': dicom_id,
                'img_original_path': img_original_path,
                'img_256_path': img_256_path,
                'disease': disease,
                'exists': exists,
            })

            # 增加 seed
            seed_counter += 1

# 将拆分后的数据转换为 DataFrame
expanded_df = pd.DataFrame(expanded_rows)

# 5. 保存为 Parquet 文件
expanded_df.to_parquet('/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full_v2.parquet', index=False)

print("数据拆分并格式化成功，已保存为 Parquet 文件！")


合并前的行数: 377110
合并后的行数: 377110
数据拆分并格式化成功，已保存为 Parquet 文件！


In [13]:
import pandas as pd

def preview_parquet(path, n=10):
    """
    读取指定 parquet 文件并打印前 n 行

    参数:
        path (str): parquet 文件路径
        n (int): 打印的行数，默认 5
    """
    try:
        df = pd.read_parquet(path)
        print("列名：", list(df.columns))
        print("\n前几行记录：")
        print(df.head(n))
        print(f"\n总行数: {len(df)}")
        return df
    except Exception as e:
        print(f"读取 parquet 文件失败: {e}")
        return None

# 使用示例
df = preview_parquet("/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full_v2.parquet", n=20)
# df_sub1 = df["exists"][:30]
# df_sub2 = df["disease"][:30]
# df_sub1, df_sub2


列名： ['seed', 'data_source', 'prompt', 'extra_info', 'dicom_id', 'img_original_path', 'img_256_path', 'disease', 'exists']

前几行记录：
    seed data_source                             prompt  \
0      0    cxr_crop  [{'content': '', 'role': 'user'}]   
1      1    cxr_crop  [{'content': '', 'role': 'user'}]   
2      2    cxr_crop  [{'content': '', 'role': 'user'}]   
3      3    cxr_crop  [{'content': '', 'role': 'user'}]   
4      4    cxr_crop  [{'content': '', 'role': 'user'}]   
5      5    cxr_crop  [{'content': '', 'role': 'user'}]   
6      6    cxr_crop  [{'content': '', 'role': 'user'}]   
7      7    cxr_crop  [{'content': '', 'role': 'user'}]   
8      8    cxr_crop  [{'content': '', 'role': 'user'}]   
9      9    cxr_crop  [{'content': '', 'role': 'user'}]   
10    10    cxr_crop  [{'content': '', 'role': 'user'}]   
11    11    cxr_crop  [{'content': '', 'role': 'user'}]   
12    12    cxr_crop  [{'content': '', 'role': 'user'}]   
13    13    cxr_crop  [{'content': '', 'role

In [35]:
import pandas as pd
import cv2
import time, random

df = pd.read_parquet('/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full_v2.parquet')
paths = df['img_original_path'].unique().tolist()
sample_paths = random.sample(paths, min(10, len(paths)))

print(sample_paths[:5])

start = time.time()
for p in sample_paths:
    img = cv2.imread(p)
    if img is not None:
        _ = cv2.resize(img, (256, 256))
total_sample_time = time.time() - start
avg_per_image = total_sample_time / len(sample_paths)
total_images = len(paths)
est_single = avg_per_image * total_images
est_96 = est_single / 96

results = {
    'sample_count': len(sample_paths),
    'total_images': total_images,
    'total_time_sample_s': total_sample_time,
    'avg_time_per_image_s': avg_per_image,
    'estimated_time_single_s': est_single,
    'estimated_time_96cores_s': est_96
}
print(pd.DataFrame.from_dict(results, orient='index', columns=['value']))


['/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files/p11/p11063065/s53033221/d6988264-ef9b1993-bc183ebf-38960313-54d10c43.jpg', '/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files/p19/p19998330/s58626532/6c6f0868-10028db0-0270ff74-65210a42-e084c544.jpg', '/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files/p14/p14767213/s54407749/8c83b1d3-37ab9008-4a3a5056-89bac880-95d016c7.jpg', '/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files/p18/p18268875/s52525965/ea78df92-50c0d7b1-d29849e2-2704c06e-89a66f5f.jpg', '/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files/p17/p17477304/s52291282/02974fed-1605f6cb-99757fcd-a573d926-7a858725.jpg']
                                  value
sample_count                  10.000000
total_images              242798.000000
total_time_sam

In [20]:
import os
import pandas as pd
import cv2
import multiprocessing
from tqdm import tqdm

# 读取 CSV 文件
csv_file = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold_data/fold0/fold0_train_subset_10000.csv'
df = pd.read_csv(csv_file)

# 获取唯一的原始路径列表
unique_paths = df['path'].unique().tolist()

# 替换路径前缀
prefix = "/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files"
prefix_256 = "/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/files_256"

def process_image_path(original_path):
    new_path = original_path.replace(prefix, prefix_256)

    # 创建目标文件夹
    new_dir = os.path.dirname(new_path)
    # if not os.path.exists(new_dir):
    os.makedirs(new_dir, exist_ok=True)

    # 检查原始文件是否存在
    if not os.path.exists(original_path):
        print(f"Warning: 原始文件不存在: {original_path}")
    
    # 读取图像并进行 resize
    img = cv2.imread(original_path)
    if img is not None:
        # Resize 图像为 256x256
        resized_img = cv2.resize(img, (256, 256))

        # 检查新文件是否已存在，如果存在，打印警告
        if os.path.exists(new_path):
            print(f"Warning: 新文件已存在并将被替换: {new_path}")
        
        # 保存 resized 图像
        cv2.imwrite(new_path, resized_img)
    
    return new_path  # 返回新路径以便在进程完成后追踪

def parallel_process(unique_paths, num_processes=96):
    # 使用 tqdm 显示进度条
    with multiprocessing.Pool(processes=num_processes) as pool:
        results = list(tqdm(pool.imap(process_image_path, unique_paths), total=len(unique_paths)))
    return results

# 开始处理并显示进度
new_paths = parallel_process(unique_paths)

print("任务完成，所有图像已保存到新的路径！")


  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 1/10000 [00:00<1:21:52,  2.04it/s]

  0%|          | 19/10000 [00:00<04:18, 38.57it/s] 

  1%|          | 97/10000 [00:00<00:50, 195.45it/s]

  1%|▏         | 130/10000 [00:00<00:50, 196.83it/s]

...

 98%|█████████▊| 9793/10000 [00:32<00:00, 480.02it/s]

 98%|█████████▊| 9844/10000 [00:32<00:00, 379.90it/s]

 99%|█████████▉| 9914/10000 [00:32<00:00, 440.58it/s]

100%|██████████| 10000/10000 [00:32<00:00, 304.05it/s]


任务完成，所有图像已保存到新的路径！


In [15]:
import pandas as pd

# 读取 CSV 文件和 full.parquet 文件

subset = "20000"

train_csv = f'/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold_data/fold0/fold0_train_subset_{subset}.csv'
full_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full_v2.parquet'

# 读取数据
train_df = pd.read_csv(train_csv)
full_df = pd.read_parquet(full_parquet)

# 在 full.parquet 中查找与 train 数据中 path 对应的记录
train_paths = train_df['path'].unique().tolist()
print(f"在 train 数据中找到 {len(train_paths)} 个唯一的路径。")
train_subset = full_df[full_df['img_original_path'].isin(train_paths)]

print(f"找到 {len(train_subset)} 条记录与 train 数据中的路径匹配。")

# 更新数据，替换 seed 和 parquet_path
train_subset['data_source'] = 'cxr_crop'
train_subset['parquet_path'] = 'data/cxr_all/full.parquet'
train_subset['prompt'] = train_subset.apply(lambda row: [{"content": "", "role": "user"}], axis=1)
train_subset['extra_info'] = train_subset.apply(lambda row: {
    "env_config": {"parquet_path": "data/cxr_all/full.parquet"},
    "env_name": "cxr_crop",
    "seed": row['seed'],  # 使用 full.parquet 中的 seed
    "split": "train"
}, axis=1)

# 创建最终需要的列格式
final_df = train_subset[[
    'data_source', 'prompt', 'extra_info'
]]

# 保存为新的 parquet 文件
output_parquet = f'/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold_data/fold0/fold0_train_subset_{subset}_parquet.parquet'
final_df.to_parquet(output_parquet, index=False)

print("任务完成，train 数据已成功保存为新的 parquet 文件！")


在 train 数据中找到 20000 个唯一的路径。
找到 31700 条记录与 train 数据中的路径匹配。


/tmp/ipykernel_92839/4196079149.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_subset['data_source'] = 'cxr_crop'
/tmp/ipykernel_92839/4196079149.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_subset['parquet_path'] = 'data/cxr_all/full.parquet'
/tmp/ipykernel_92839/4196079149.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.p

任务完成，train 数据已成功保存为新的 parquet 文件！


/tmp/ipykernel_92839/4196079149.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_subset['extra_info'] = train_subset.apply(lambda row: {


In [2]:
import pandas as pd

# 读取 CSV 文件和 full.parquet 文件
train_csv = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold_data/fold0/fold0_train_subset_10000.csv'
full_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full_v2.parquet'

# 读取数据
train_df = pd.read_csv(train_csv)
full_df = pd.read_parquet(full_parquet)

# 获取 train 数据中所有的路径
train_paths = train_df['path'].tolist()

# 创建一个空列表来存储结果
missing_records = []

# 检查每个路径是否在 full_df 中存在
for path in train_paths:
    if path in full_df['img_original_path'].values:
        missing_records.append({'path': path, 'found': 'Yes'})
    else:
        missing_records.append({'path': path, 'found': 'No'})

# 将结果转换为 DataFrame
missing_df = pd.DataFrame(missing_records)

# 保存到一个新的 CSV 文件，以便查看
missing_df.to_csv('/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/fold_data/fold0/missing_paths.csv', index=False)

# 打印前五条记录
print(missing_df.head())



KeyboardInterrupt: 

In [18]:
import os
import pandas as pd

# ===== 输入/输出路径配置 =====
ms_csv = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/MS_CXR_combined.csv'
full_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full.parquet'
# MS_CXR_combined.csv 的 path 前要加的前缀（注意，这里不重复加 'files'；CSV 里的 path 已以 files/... 开头）
PREFIX = '/mnt/input/mimic_cxr_jpg/ori_dataset/mimic-cxr-jpg-2.1.0/mimic-cxr-jpg-2.1.0.physionet.org/'
# 产出 parquet 路径
out_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/ms_cxr_all.parquet'
# 未匹配路径导出
out_unmatched_csv = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/ms_cxr_all_unmatched.csv'
# ===== 目标 extra_info 配置 =====
TARGET_PARQUET_PATH = 'data/cxr_all/full.parquet'
SPLIT = 'test'
# =================================

# 1) 读取 MS_CXR，并按 path 去重 -> 构造完整 img_original_path
ms_df = pd.read_csv(ms_csv, dtype=str)
ms_df['path'] = ms_df['path'].astype(str)
uniq_paths = ms_df['path'].dropna().drop_duplicates().reset_index(drop=True)
uniq_full_paths = PREFIX.rstrip('/') + '/' + uniq_paths.str.lstrip('/')
uniq_full_paths.name = 'img_original_path'  # 命名一下列

print(f'Unique paths from MS_CXR: {len(uniq_full_paths)}')

# 2) 读取 full.parquet，并构建：img_original_path -> seeds[]
full_df = pd.read_parquet(full_parquet)
# 防御：只取需要的两列，且去重
full_min = full_df[['img_original_path', 'seed']].dropna().drop_duplicates()
# 生成 mapping：路径 -> 多个 seed（去重）
seed_map = (full_min.groupby('img_original_path')['seed']
            .apply(lambda s: sorted(set(int(x) for x in s)))
            .to_dict())

# 3) 为每个唯一路径生成多条记录（每个 seed 一条）
records = []
unmatched = []

for p in uniq_full_paths:
    seeds = seed_map.get(p)
    if not seeds:
        unmatched.append(p)
        continue
    for sd in seeds:
        records.append({
            'data_source': 'cxr_crop',
            'prompt': [{"content": "", "role": "user"}],
            'extra_info': {
                'env_config': {'parquet_path': TARGET_PARQUET_PATH},
                'env_name': 'cxr_crop',
                'seed': int(sd),
                'split': SPLIT
            }
        })

# 4) 写出结果
out_df = pd.DataFrame.from_records(records)
out_df.to_parquet(out_parquet, index=False)
print(f'Wrote parquet: {out_parquet}  | rows: {len(out_df)}')

# 5) 导出未匹配路径，便于排查
if unmatched:
    pd.DataFrame({'img_original_path': unmatched}).to_csv(out_unmatched_csv, index=False)
    print(f'Unmatched paths: {len(unmatched)} -> {out_unmatched_csv}')
else:
    print('All unique paths matched seeds in full.parquet.')


Unique paths from MS_CXR: 1047
Wrote parquet: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/ms_cxr_all.parquet  | rows: 2416
Unmatched paths: 68 -> /home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/ms_cxr_all_unmatched.csv
